In [1]:
import pandas as pd
from pathlib import Path

DATA = Path.home() / "datasets" / "apache-jira" / "issues.csv"
print("File exists:", DATA.exists())
print("Size (GB):", round(DATA.stat().st_size / 1e9, 2))



File exists: True
Size (GB): 1.91


In [2]:
cols = [
    "key", "created", "resolutiondate",
    "priority.name", "status.name", "statusCategory.name",
    "resolution.name", "issuetype.name",
    "project.key", "project.name",
]

df = pd.read_csv(DATA, usecols=cols)
print(df.shape)
df.head()

(1149323, 10)


,key,resolution.name,priority.name,status.name,statusCategory.name,issuetype.name,project.key,project.name,resolutiondate,created
0,WW-712,Fixed,Minor,Closed,Done,Improvement,WW,Struts 2,2005-01-01 07:50:46,2005-01-01 07:47:50
1,XALANC-446,Fixed,Blocker,Resolved,Done,Bug,XALANC,XalanC,2004-12-30 05:30:36,2004-12-25 22:50:30
2,ROL-587,Cannot Reproduce,Critical,Closed,Done,Bug,ROL,Apache Roller,2005-01-02 15:21:00,2005-01-01 13:52:46
3,AXIS-1741,NaN,Major,Open,To Do,Bug,AXIS,Axis,NaN,2005-01-02 19:13:37
4,AXIS-1745,NaN,Major,Open,To Do,Bug,AXIS,Axis,NaN,2005-01-03 03:34:52


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1149323 entries, 0 to 1149322
Data columns (total 10 columns):
 #   Column               Non-Null Count    Dtype
---  ------               --------------    -----
 0   key                  1149323 non-null  str  
 1   resolution.name      946655 non-null   str  
 2   priority.name        1117464 non-null  str  
 3   status.name          1131224 non-null  str  
 4   statusCategory.name  1131224 non-null  str  
 5   issuetype.name       1131224 non-null  str  
 6   project.key          1131224 non-null  str  
 7   project.name         1131224 non-null  str  
 8   resolutiondate       946715 non-null   str  
 9   created              1131224 non-null  str  
dtypes: str(10)
memory usage: 87.7 MB


In [4]:
df["created"] = pd.to_datetime(df["created"], errors="coerce")
df["resolutiondate"] = pd.to_datetime(df["resolutiondate"], errors="coerce")

df[["created", "resolutiondate"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 1149323 entries, 0 to 1149322
Data columns (total 2 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   created         1130935 non-null  datetime64[us]
 1   resolutiondate  946715 non-null   datetime64[us]
dtypes: datetime64[us](2)
memory usage: 17.5 MB


In [5]:
df["resolution_days"] = (df["resolutiondate"] - df["created"]).dt.total_seconds() / 86400

df["resolution_days"].describe()

count    946426.000000
mean        203.954379
std         520.375742
min          -0.372951
25%           1.049771
50%          11.915307
75%         112.017925
max        8001.517257
Name: resolution_days, dtype: float64

In [6]:
(df["resolution_days"] < 0).sum()

np.int64(18)

In [7]:
resolved = df[df["resolution_days"] >= 0].copy()
print("Resolved issues for analysis:", len(resolved))
print("Dropped (negative):", (df["resolution_days"] < 0).sum())

Resolved issues for analysis: 946408
Dropped (negative): 18


In [8]:
resolved.groupby("priority.name")["resolution_days"].median().sort_values()

priority.name
Trivial             4.576389
P0                  4.814670
Urgent              6.900596
Blocker             7.770035
Low                 9.627245
Major              10.938970
P4                 11.450741
Critical           11.963102
High               12.830822
Normal             13.292616
Minor              14.471557
P1                 28.692049
P2                 28.863605
P3                 98.932072
Not a Priority    746.897992
Name: resolution_days, dtype: float64

In [9]:
resolved["project.name"].value_counts()

project.name
Spark                                   44867
Apache Flex                             31565
Flink                                   30386
HBase                                   26611
Ambari                                  23649
                                        ...  
eSCIMo                                      1
Axis2 Transports                            1
ActiveMQ Website                            1
COTTON                                      1
Maven JLink Plugin (Moved to GitHub)        1
Name: count, Length: 631, dtype: int64

In [10]:
resolved["priority.name"].value_counts()

priority.name
Major             620202
Minor             179388
Critical           44155
Blocker            34793
Trivial            27202
Normal             11227
P2                  6390
Low                 6164
P3                  2629
P1                   721
Urgent               642
Not a Priority       354
P0                   274
P4                   237
High                 156
Name: count, dtype: int64

In [11]:
# For each project, get the set of distinct priorities it uses
priority_menus = resolved.groupby("project.name")["priority.name"].unique()

# Look at a few big projects' menus
for proj in ["Spark", "Apache Flex", "Flink", "HBase", "Ambari"]:
    print(proj, "->", sorted(priority_menus[proj]))

Spark -> ['Blocker', 'Critical', 'Major', 'Minor', 'Trivial']
Apache Flex -> ['Blocker', 'Critical', 'Major', 'Minor', 'Trivial']
Flink -> ['Blocker', 'Critical', 'Major', 'Minor', 'Not a Priority']
HBase -> ['Blocker', 'Critical', 'Major', 'Minor', 'Trivial']
Ambari -> ['Blocker', 'Critical', 'Major', 'Minor', 'Trivial']


In [12]:
# How many projects include each of the four core priorities?
core = ["Blocker", "Critical", "Major", "Minor"]
for p in core:
    n = resolved[resolved["priority.name"] == p]["project.name"].nunique()
    print(f"{p}: used by {n} projects")

Blocker: used by 484 projects
Critical: used by 501 projects
Major: used by 614 projects
Minor: used by 567 projects


In [13]:
df.shape

(1149323, 11)

In [14]:
core_resolved = resolved[resolved["priority.name"].isin(core)]
core_resolved.groupby("priority.name")["resolution_days"].median().sort_values()

priority.name
Blocker      7.770035
Major       10.938970
Critical    11.963102
Minor       14.471557
Name: resolution_days, dtype: float64

In [15]:
by_project = (
    core_resolved
    .groupby(["project.name", "priority.name"])["resolution_days"]
    .median()
)
by_project

project.name            priority.name
ACE                     Blocker          109.879583
                        Critical         176.719878
                        Major            202.574722
                        Minor            310.856713
ALOIS                   Major             14.949213
                                            ...    
serf                    Major             33.434491
                        Minor            112.320532
www.apache.org website  Critical           0.810197
                        Major              0.948194
                        Minor            260.161939
Name: resolution_days, Length: 2166, dtype: float64

In [16]:
project_priority = (
    core_resolved
    .groupby(["project.name", "priority.name"])["resolution_days"]
    .agg(["median", "count"])
)
project_priority

median  count
project.name           priority.name                   
ACE                    Blocker        109.879583      7
                       Critical       176.719878     12
                       Major          202.574722    395
                       Minor          310.856713     73
ALOIS                  Major           14.949213      1
...                                          ...    ...
serf                   Major           33.434491    156
                       Minor          112.320532      9
www.apache.org website Critical         0.810197      1
                       Major            0.948194      8
                       Minor          260.161939      2

[2166 rows x 2 columns]

In [17]:
reliable = project_priority[project_priority["count"] >= 30]
reliable

median  count
project.name priority.name                   
ACE          Major          202.574722    395
             Minor          310.856713     73
AMATERASU    Major          108.133056     47
Abdera       Major            3.696464    176
             Minor            3.578137     56
...                                ...    ...
jclouds      Major           13.267396    913
             Minor            7.365156    170
mod_python   Major           25.888032    121
             Minor           34.466215     44
serf         Major           33.434491    156

[1159 rows x 2 columns]

In [18]:
wide = reliable["median"].unstack()
wide.head(10)

priority.name,Blocker,Critical,Major,Minor
project.name,,,,
ACE,NaN,NaN,202.574722,310.856713
AMATERASU,NaN,NaN,108.133056,NaN
Abdera,NaN,NaN,3.696464,3.578137
Accumulo,4.091019,13.361117,26.886262,28.753785
ActiveMQ .Net,NaN,23.233669,8.911400,7.139352
ActiveMQ Apollo (Retired),NaN,NaN,4.191887,21.441481
ActiveMQ Artemis,15.871424,26.156047,15.614884,19.486678
ActiveMQ C++ Client,NaN,6.814884,2.634988,1.239850
ActiveMQ Classic,15.516574,32.518403,11.376470,18.144618


In [19]:
mm = wide.dropna(subset=["Major", "Minor"])
print("Projects with both Major and Minor:", len(mm))

Projects with both Major and Minor: 361


In [20]:
major_faster = (mm["Major"] < mm["Minor"]).mean()
print(f"Major resolves faster than Minor in {major_faster:.1%} of projects")

Major resolves faster than Minor in 58.4% of projects


In [21]:
core_resolved["project.name"].value_counts().head(10)

project.name
Spark            43196
Apache Flex      30966
Flink            30038
HBase            25312
Ambari           23555
Camel            20552
Hive             20057
Ignite           17816
Apache Arrow     15094
Hadoop Common    14153
Name: count, dtype: int64

In [22]:
spark = core_resolved[core_resolved["project.name"] == "Spark"]

spark.groupby("priority.name")["resolution_days"].agg(["median", "count"])

,median,count
priority.name,,
Blocker,3.862535,1421
Critical,9.807280,1643
Major,6.311343,29445
Minor,5.604005,10687


In [23]:
spark = core_resolved[core_resolved["project.name"] == "Spark"]

spark.groupby("priority.name")["resolution_days"].agg(["mean", "count"])

,mean,count
priority.name,,
Blocker,24.015555,1421
Critical,114.473250,1643
Major,132.240581,29445
Minor,145.011621,10687


In [24]:
spark.groupby("priority.name")["resolution_days"].quantile([0.25, 0.5, 0.75])

priority.name      
Blocker        0.25     0.870023
               0.50     3.862535
               0.75    15.715081
Critical       0.25     1.441858
               0.50     9.807280
               0.75    69.202980
Major          0.25     0.940417
               0.50     6.311343
               0.75    56.441262
Minor          0.25     0.946076
               0.50     5.604005
               0.75    64.061528
Name: resolution_days, dtype: float64

In [25]:
q = spark.groupby("priority.name")["resolution_days"].quantile([0.25, 0.75]).unstack()
q.columns = ["Q1", "Q3"]
q["IQR"] = q["Q3"] - q["Q1"]
q

,Q1,Q3,IQR
priority.name,,,
Blocker,0.870023,15.715081,14.845058
Critical,1.441858,69.202980,67.761123
Major,0.940417,56.441262,55.500845
Minor,0.946076,64.061528,63.115451


In [26]:
from pathlib import Path

out = Path.home() / "data-engineer-prep" / "python" / "jira-analysis" / "outputs"
out.mkdir(exist_ok=True)

# 1. Per-project median by priority (within-project comparison source)
wide.to_csv(out / "median_by_project_priority.csv")

# 2. Spark quartile / IQR table (the spread finding)
q.to_csv(out / "spark_quartiles.csv")

# 3. Spark median + count ladder (the inversion finding)
spark.groupby("priority.name")["resolution_days"].agg(["median", "count"]).to_csv(out / "spark_priority_ladder.csv")

print("Exported to:", out)
print(list(out.iterdir()))

Exported to: /home/jkzero/data-engineer-prep/python/jira-analysis/outputs
[PosixPath('/home/jkzero/data-engineer-prep/python/jira-analysis/outputs/median_by_project_priority.csv'), PosixPath('/home/jkzero/data-engineer-prep/python/jira-analysis/outputs/spark_quartiles.csv'), PosixPath('/home/jkzero/data-engineer-prep/python/jira-analysis/outputs/spark_priority_ladder.csv')]


In [27]:
from pathlib import Path
out = Path.home() / "data-engineer-prep" / "python" / "jira-analysis" / "outputs"

# Ladder: priority is the index → name it on reset
(spark.groupby("priority.name")["resolution_days"]
      .agg(["median", "count"])
      .rename_axis("priority")
      .to_csv(out / "spark_priority_ladder.csv"))

# Quartiles: same — priority is the index
q.rename_axis("priority").to_csv(out / "spark_quartiles.csv")

# Per-project: project is the index
wide.rename_axis("project").to_csv(out / "median_by_project_priority.csv")

print("Re-exported with clean names")

Re-exported with clean names


In [29]:
out = Path.home() / "data-engineer-prep" / "python" / "jira-analysis" / "outputs"

with pd.ExcelWriter(out / "jira_spark_analysis.xlsx") as writer:
    (spark.groupby("priority.name")["resolution_days"]
          .agg(["median", "count"])
          .rename_axis("priority")
          .to_excel(writer, sheet_name="priority_ladder"))
    q.rename_axis("priority").to_excel(writer, sheet_name="quartiles")
    wide.rename_axis("project").to_excel(writer, sheet_name="project_medians")

print("done:", (out / "jira_spark_analysis.xlsx").exists())

done: True
